# 経路選択モデル演習 — Recursive Logit Model

松山市の歩行者 GPS トラッキングデータ（マップマッチング済み）を使い，
**Recursive Logit (RL) モデル**（Fosgerau et al., 2013）を推定します．

**演習の流れ**
1. [セットアップ](#setup) — パッケージのインストールと設定読み込み
2. [Part 1: ネットワーク可視化](#part1) — 道路ネットワークと経路データの確認
3. [Part 2: リンク説明変数の基礎集計](#part2) — 土地利用・道路属性の分布・相関分析
4. [Part 3: デフォルト設定での推定](#part3) — RL モデルの推定とサマリー表示
5. [Part 4: SAM3 による属性追加と再推定](#part4) — 航空写真からセグメント割合を算出し追加説明変数として使用

---

**参考文献**: Fosgerau, M., Frejinger, E., & Karlstrom, A. (2013). A link based network route choice model with unrestricted choice set. *Transportation Research Part B*, 56, 70–80.

## セットアップ <a id="setup"></a>

In [ ]:
# Google Drive マウント（必要な場合）
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Google Drive マウント（必要な場合）
# import os; os.chdir('path/to/traffic_class_rcm_2026') # Google Drive 内のプロジェクトディレクトリに移動
# import sys; sys.path.append("src")

# Google Colab でのパッケージインストール（ローカル実行時はスキップ可）
# !pip install uv -q
# !uv pip install -e . --system -q

In [ ]:
import warnings
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib_fontja  # 日本語フォント設定
import seaborn as sns

from rcm import RecursiveLogit, load_road_network, load_trips, load_link_features
from rcm.visualize import visualize_network, visualize_route

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# 設定ファイルの読み込み
with open('config.yaml', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print('設定ファイル読み込み完了')
print(yaml.dump(config, allow_unicode=True))

---

## Part 1: ネットワーク可視化 <a id="part1"></a>

松山市の道路ネットワークと，マップマッチング済みの歩行者経路データを読み込み，地図上で可視化します．

In [ ]:
# ネットワーク読み込み
network = load_road_network(
    config['data']['network_link'],
    config['data']['network_node'],
)

print(f'ノード数:   {len(network.nodes):,}')
print(f'リンク数:   {len(network.links):,}')

# 歩行者リンクの数
ped_links = [lk for lk in network.links if lk.ped]
print(f'歩行者リンク数: {len(ped_links):,}')

# リンク長の基本統計
lengths = [lk.length_m for lk in network.links]
print(f'リンク長 (m): 平均={np.mean(lengths):.1f}, 中央値={np.median(lengths):.1f}, 最大={max(lengths):.1f}')

In [ ]:
# 経路データ読み込み
trips = load_trips(config['data']['routes'])

print(f'経路数: {len(trips)}')
print(f'経路長 (リンク数): 平均={np.mean([len(t.chosen_route.link_ids) for t in trips]):.1f}')

# 信頼度の分布
confidences = [t.chosen_route.confidence for t in trips]
print(f'マップマッチング信頼度: 平均={np.mean(confidences):.3f}, 最小={min(confidences):.3f}')

In [ ]:
# ネットワーク可視化
# 緑: 歩行者リンク, 灰: 車両リンク
# 青丸: 経路の起点, 赤丸: 経路の終点
m = visualize_network(network, trips=trips)
m

In [ ]:
# 経路長の分布
route_lengths = [len(t.chosen_route.link_ids) for t in trips]
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(route_lengths, bins=20, edgecolor='white')
axes[0].set_xlabel('経路リンク数')
axes[0].set_ylabel('頻度')
axes[0].set_title('経路長の分布')

axes[1].hist(confidences, bins=20, edgecolor='white', color='orange')
axes[1].set_xlabel('マップマッチング信頼度')
axes[1].set_ylabel('頻度')
axes[1].set_title('信頼度の分布')
plt.tight_layout()
plt.show()

---

## Part 2: リンク説明変数の基礎集計 <a id="part2"></a>

`dlink_features.parquet` には各リンクの土地利用割合（14 カテゴリ）と道路属性（車線数・歩道幅員）が含まれます．

| 列名 | tochi_CD | 土地利用の意味 |
|------|----------|----------------|
| `landuse_1` | 01 | 田 |
| `landuse_2` | 02 | 畑 |
| `landuse_5` | 05 | 山林 |
| `landuse_6` | 06 | 水面 |
| `landuse_7` | 07 | その他の自然地 |
| `landuse_9` | 09 | 住宅用地 |
| `landuse_10` | 10 | 商業用地 |
| `landuse_11` | 11 | 工業用地 |
| `landuse_12` | 12 | 公共施設用地 |
| `landuse_13` | 13 | 公共施設用地（学校） |
| `landuse_14` | 14 | 公共施設用地（神社・寺） |
| `landuse_16` | 16 | 交通施設用地 |
| `landuse_17` | 17 | 公共空地 |
| `landuse_19` | 19 | その他の空地 |

値は国土数値情報（H20）の土地利用細分メッシュデータを各リンクの **50m バッファ** で集計した面積比率を min-max 正規化したものです（`image-link-rcm/data/processed/link_features.parquet` から取得）．  
コード番号が欠番（0, 3, 4, 8, 15, 18）のカテゴリは松山市の対象エリアに存在しないため除外されています．

In [ ]:
# リンク説明変数の読み込み
dlink = pd.read_parquet(config['data']['dlink_features'])
print(f'行数: {len(dlink)}, 列数: {len(dlink.columns)}')
print('列名:', list(dlink.columns))
dlink.head()

In [ ]:
# 基本統計量
# config の features.dlink_cols が指定されていればその列のみ、なければ全列を使用
_dlink_cols_cfg = config.get('features', {}).get('dlink_cols')
if _dlink_cols_cfg:
    feature_cols = [c for c in _dlink_cols_cfg if c in dlink.columns]
else:
    feature_cols = [c for c in dlink.columns if c != 'link_id']

print(f'使用する説明変数 ({len(feature_cols)} 列): {feature_cols}')
dlink[feature_cols].describe().round(4)

In [ ]:
# 相関行列ヒートマップ
corr = dlink[feature_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title('リンク説明変数の相関行列')
plt.tight_layout()
plt.show()

**【補足】土地利用変数のスケールについて**

- 各列は **min-max 正規化済み**（0〜1）です．最大値が 0 の列（松山市の対象エリアに存在しないカテゴリ）は除外されています．
- `landuse_9`（住宅用地）・`landuse_10`（商業用地）は 90% 以上のリンクでゼロでなく，都市部のリンクを中心に分布します．
- `landuse_1`（田）・`landuse_2`（畑）などの農地系は約 4〜6% のリンクにのみ存在し，分布が極端に歪んでいます．RL 推定では内部で z-score 正規化が行われるため，推定時のスケールは自動的に調整されます．

In [ ]:
# 土地利用割合の分布
landuse_cols = [c for c in feature_cols if c.startswith('landuse_')]
attr_cols = [c for c in feature_cols if not c.startswith('landuse_')]

if landuse_cols:
    n_lc = len(landuse_cols)
    ncols = 5
    nrows = (n_lc + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3 * nrows))
    axes_flat = axes.flatten() if nrows * ncols > 1 else [axes]
    for i, col in enumerate(landuse_cols):
        axes_flat[i].hist(dlink[col].dropna(), bins=30, edgecolor='none', alpha=0.8)
        axes_flat[i].set_title(col)
        axes_flat[i].set_xlabel('面積割合')
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)
    plt.suptitle('土地利用カテゴリ別の面積割合分布（min-max 正規化後）')
    plt.tight_layout()
    plt.show()
else:
    print('土地利用変数は使用しない設定です（dlink_cols に landuse_* なし）')

In [ ]:
# 道路属性の分布
fig, axes = plt.subplots(1, len(attr_cols), figsize=(4 * len(attr_cols), 3))
if len(attr_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, attr_cols):
    ax.hist(dlink[col].dropna(), bins=30, edgecolor='none', alpha=0.8, color='steelblue')
    ax.set_title(col)
    ax.set_xlabel('値')
plt.suptitle('道路属性の分布')
plt.tight_layout()
plt.show()

In [ ]:
# ネットワークのリンク順に特徴量を並べ替え（EDA 用）
link_ids_ordered = [lk.link_id for lk in network.links]
dlink_indexed = dlink.set_index('link_id')

# ネットワークに存在するリンクのみ使用（欠損は 0 埋め）
dlink_aligned = dlink_indexed.reindex(link_ids_ordered).fillna(0.0)
print(f'特徴量行列のサイズ: {dlink_aligned[feature_cols].shape}')
print(f'  ({len(network.links)} リンク × {len(feature_cols)} 説明変数)')

---

## Part 3: デフォルト設定での RL 推定 <a id="part3"></a>

デフォルト説明変数（リンク長 + `dlink_features.parquet` の属性）で Recursive Logit モデルを推定します．

### 推定結果の読み方

**① t 値（t-value）**

$$t_k = \frac{\hat{\beta}_k}{\text{SE}(\hat{\beta}_k)}$$

パラメータ $\hat{\beta}_k$ が統計的に 0 と異なるかを検定する値です．サンプルサイズが十分大きいとき，帰無仮説（$\beta_k = 0$）のもとで $t_k$ は標準正規分布 $N(0,1)$ に従います．

| \|t\| の目安 | 両側 p 値 | 有意水準 |
|-------------|----------|----------|
| > 1.96 | < 0.05 | * |
| > 2.58 | < 0.01 | ** |
| > 3.29 | < 0.001 | *** |

サマリー表の右端の `*` 記号がこの有意水準を示しています．

---

**② 尤度比指標（McFadden の $\rho^2$）**

$$\rho^2 = 1 - \frac{\text{LL}(*)}{\text{LL}(0)}, \qquad \overline{\rho}^2 = 1 - \frac{\text{LL}(*) - K}{\text{LL}(0)}$$

| 記号 | 意味 |
|------|------|
| $\text{LL}(0)$ | 全パラメータ = 0（等確率選択）のときの対数尤度 |
| $\text{LL}(*)$ | 推定済みパラメータの対数尤度（最大値） |
| $K$ | 推定パラメータ数 |

$\rho^2$ は線形回帰の $R^2$ に相当するモデル適合度指標です（必ず $0 \le \rho^2 \le 1$）．経路選択モデルでは **0.2〜0.4** 程度で良好な適合とされます．$\overline{\rho}^2$（調整済み）はパラメータ数によるペナルティを加えた値です．

---

**③ 精度（Precision）と ④ F1 スコア**

経路予測の評価はリンク集合の一致を基に定義します．予測経路のリンク集合を $\hat{R}$，実際の経路のリンク集合を $R$ とすると：

$$\text{Precision} = \frac{|\hat{R} \cap R|}{|\hat{R}|}, \qquad \text{Recall} = \frac{|\hat{R} \cap R|}{|R|}$$

$$\text{F1} = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

$$\text{ROR} = \frac{|\hat{R} \cap R|}{|\hat{R} \cup R|}$$

| 指標 | 何を測るか |
|------|-----------|
| Precision（精度） | 予測リンクのうち実際に通行したリンクの割合 |
| Recall（再現率） | 実際のリンクのうち予測で拾えたリンクの割合 |
| F1 スコア | Precision と Recall の調和平均（バランス指標） |
| ROR（Route Overlap Ratio） | 和集合に対する共通リンクの割合（Jaccard 係数と等価） |

下のセルでは代表経路の ROR を確認します．F1 や Precision/Recall を全経路で集計する場合は，同様のロジックを `trips` 全体に適用してください．

---

**⑤ AIC（赤池情報量規準）**

$$\text{AIC} = -2 \cdot \text{LL}(*) + 2K$$

「当てはまりの良さ」と「モデルの複雑さ（パラメータ数）」のトレードオフを一つの数値で表す指標です．

- $-2\,\text{LL}(*)$ が小さい ＝ 当てはまりが良い → AIC を下げる方向
- $2K$ はパラメータ数へのペナルティ → 変数を増やすと AIC が上がる

**AIC の使い方:** 複数モデルを比較する際に用い，**値が小さいモデルが優れている**と判断します．$\rho^2$ との違いは以下のとおりです．

| | $\rho^2$ | AIC |
|---|---|---|
| 範囲 | 0〜1 | 制限なし |
| 使い方 | 単体でも解釈可 | モデル間の比較のみ |
| パラメータペナルティ | $\overline{\rho}^2$ で対応 | 構造に組み込み済み |

In [ ]:
# 説明変数の準備（config から使用列を取得してフィーチャー行列を作成）
dlink_cols = config.get('features', {}).get('dlink_cols')
extra_features, feature_cols = load_link_features(
    config['data']['dlink_features'],
    link_ids_ordered,
    dlink_cols,
)
print(f'使用する説明変数 ({len(feature_cols)} 列): {feature_cols}')
print(f'  (config の features.dlink_cols で変更可能)')
print()
print('説明変数:')
print('  length_m  (リンク長 — 常に含まれる)')
for name in feature_cols:
    print(f'  {name}')
print(f'  合計 {1 + len(feature_cols)} 変数 + gamma (割引率)')

In [ ]:
# モデル推定（数分かかる場合があります）
model = RecursiveLogit(
    max_iter=config['model']['max_iter'],
    conv_eps=config['model']['conv_eps'],
)

print('推定中…')
model.fit(
    trips,
    network,
    extra_link_features=extra_features,
    feature_names=feature_cols,
)
print('推定完了')

In [ ]:
# 推定結果のサマリー
results_default = model.summary()
print(f'\nAIC: {model.aic:.4f}')

In [ ]:
# 経路予測の例（最初の 3 経路）
print('経路予測の例（最初の 3 経路）:')
for trip in trips[:3]:
    predicted = model.predict(trip.origin, trip.destination, network)
    actual = trip.chosen_route.link_ids
    # Route Overlap Ratio
    overlap = len(set(predicted) & set(actual)) / max(len(set(predicted) | set(actual)), 1)
    print(f'  trip={trip.trip_id}: actual={len(actual)} links, predicted={len(predicted)} links, ROR={overlap:.2f}')

In [ ]:
# 最初の経路の可視化（実際 vs 予測）
trip_ex = trips[0]
pred_ex = model.predict(trip_ex.origin, trip_ex.destination, network)
m_route = visualize_route(trip_ex, network, predicted_link_ids=pred_ex)
print(f'trip_id={trip_ex.trip_id}: 緑=実際の経路, 赤点線=予測経路')
m_route

---

## Part 4: SAM3 による属性追加と再推定 <a id="part4"></a>

Ultralytics SAM3 を使って航空写真から任意のキーワード（例: "road", "sidewalk", "tree"）の
セグメント面積割合を計算し，追加説明変数として RL モデルを再推定します．

### 設定方法
`config.yaml` を開いて `sam3.words` にキーワードを追加してください：

```yaml
sam3:
  words: ["road", "sidewalk", "tree", "building"]
```

キーワードの数だけ説明変数が追加されます．セグメンテーション結果は `cache/sam3/` にキャッシュされるため，再実行時は再計算されません．

In [ ]:
# SAM3 のインストール（初回のみ）
# !pip install uv -q
# !uv pip install ".[sam3]" --system -q

In [ ]:
# config.yaml を再読み込み（words を追加した後に実行）
with open('config.yaml', encoding='utf-8') as f:
    config = yaml.safe_load(f)

words = config['sam3']['words']
print(f'SAM3 キーワード: {words}')

if not words:
    print('\n⚠ config.yaml の sam3.words にキーワードを追加してから再実行してください．')
    print('  例: words: ["road", "sidewalk", "tree", "building"]')

In [ ]:
if words:
    from rcm.sam3 import get_sam3_features, visualize_sam3_mask
    from pathlib import Path
    import numpy as np

    # node_lookup の作成
    node_lookup = {n.node_id: n for n in network.nodes}

    print('SAM3 セグメンテーション実行中...')
    sam3_df = get_sam3_features(
        aerial_photo_path=config['data']['aerial_photo'],
        geo_json_path=config['data']['aerial_photo_geo'],
        words=words,
        links=network.links,
        node_lookup=node_lookup,
        cache_dir=config['sam3']['cache_dir'],
        model_name=config['sam3']['model'],
        buffer_px=config['sam3']['buffer_px'],
    )
    print('完了')
    print(sam3_df.describe())

In [ ]:
if words:
    # セグメント面積割合の分布（箱ひげ図）
    sam3_aligned = sam3_df.set_index('link_id').reindex(link_ids_ordered).fillna(0.0)

    fig, axes = plt.subplots(1, len(words), figsize=(4 * len(words), 4), sharey=False)
    if len(words) == 1:
        axes = [axes]
    for ax, word in zip(axes, words):
        ax.boxplot(sam3_aligned[word].values, vert=True)
        ax.set_title(f'「{word}」セグメント割合')
        ax.set_ylabel('面積割合')
        ax.set_xlabel('')
    plt.suptitle('SAM3 セグメント面積割合の分布（リンク別）')
    plt.tight_layout()
    plt.show()

In [ ]:
if words:
    # セグメンテーション結果を航空写真に重ねて可視化（最初のキーワード）
    from pathlib import Path
    first_word = words[0]
    mask_path = Path(config['sam3']['cache_dir']) / first_word / 'mask.npy'

    if mask_path.exists():
        mask = np.load(mask_path)
        composited = visualize_sam3_mask(
            config['data']['aerial_photo'],
            mask,
            word=first_word,
        )
        # サムネイル表示（大きすぎるため縮小）
        thumb_w = 800
        ratio = thumb_w / composited.width
        thumb = composited.resize((thumb_w, int(composited.height * ratio)))

        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(thumb)
        ax.set_title(f'SAM3 セグメンテーション結果: 「{first_word}」（赤色がセグメント領域）')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f'キャッシュが見つかりません: {mask_path}')

In [ ]:
if words:
    # SAM3 特徴量をデフォルト特徴量と結合して再推定
    sam3_features = sam3_aligned[words].values  # (n_links, len(words))
    combined_features = np.concatenate([extra_features, sam3_features], axis=1)
    combined_names = feature_cols + words

    print(f'特徴量次元: {extra_features.shape[1]} (デフォルト) + {sam3_features.shape[1]} (SAM3) = {combined_features.shape[1]}')

    model_sam3 = RecursiveLogit(
        max_iter=config['model']['max_iter'],
        conv_eps=config['model']['conv_eps'],
    )
    print('SAM3 特徴量追加モデルを推定中...')
    model_sam3.fit(
        trips,
        network,
        extra_link_features=combined_features,
        feature_names=combined_names,
    )
    print('推定完了')

In [ ]:
if words:
    # SAM3 モデルのサマリー
    results_sam3 = model_sam3.summary()
    print(f'\nAIC: {model_sam3.aic:.4f}')

In [ ]:
if words:
    # モデル比較
    comparison = pd.DataFrame({
        'モデル': ['デフォルト (リンク長 + dlink_features)', f'SAM3 追加 (+{words})'],
        'パラメータ数': [model.n_params, model_sam3.n_params],
        'LL(0)': [results_default['ll_null'], results_sam3['ll_null']],
        'LL(*)': [results_default['ll_final'], results_sam3['ll_final']],
        'ρ²': [results_default['rho_squared'], results_sam3['rho_squared']],
        'Adj. ρ²': [results_default['adj_rho_squared'], results_sam3['adj_rho_squared']],
        'AIC': [model.aic, model_sam3.aic],
        'gamma': [results_default['gamma'], results_sam3['gamma']],
    }).set_index('モデル')

    display(comparison.round(4))

    # LL の改善
    ll_diff = results_sam3['ll_final'] - results_default['ll_final']
    aic_diff = model_sam3.aic - model.aic
    print(f'\nΔLL: {ll_diff:+.4f}')
    print(f'ΔAIC: {aic_diff:+.4f} (負であればSAM3モデルが優れている)')